In [25]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

## [RAG절차]
1. 문서를 읽는다
    * %pip install -u -q docx2txt
2. 문서를 쪼갠다
    * %pip install -qU langchain-text-splitters
3. 쪼갠 문서를 임베딩하여 vector database에 넣음(local에 저장) cf. 클라우드에 저장
    * %pip install -q langchain-chroma
4. 질문을 이용해 유사도 검색
5. 유사도 검색한 문서를 LLM에 질문과 함꼐 전달하여 답변을 얻음(렝체인 사용 가능)
    * %pip install langchain
    * (https://www.langchain.com 에서 key 생성 .env에 LANGCHAIN_API_KEY로 추가)

# 0. 패키지 설치

In [ ]:
# 문서 읽어오기
pip install -u -q docx2txt

In [2]:
# 텍스트를 chunk로 나누는 기능만 있는 경량 모듈
%pip install -qU langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 벡터DB(로컬DB) 어제의 chromadb가 아님
%pip install -q langchain-chroma

Note: you may need to restart the kernel to use updated packages.


In [4]:
# langchain 사용
%pip install langchain


   ---------------------- ----------------- 4/7 [langgraph-prebuilt]
   ---------------------------- ----------- 5/7 [langgraph]
   ---------------------------------------- 7/7 [langchain]

Note: you may need to restart the kernel to use updated packages.


# 1. 문서읽기(X)

In [6]:
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./data/소득세법(법률)(제21065호)(20260102).docx')
document = loader.load()

In [7]:
len(document)

1

# 2. 문서를 쪼개면서 (O)
- https://docs.langchain.com/oss/python/integrations/splitters
## 2.1 1500토큰씩 쪼개서 읽어오기

In [15]:
import time
from langchain_text_splitters import TokenTextSplitter
loader = Docx2txtLoader('./data/소득세법(법률)(제21065호)(20260102).docx')
# gpt-4, gpt-4o, gpt-4 turbo, gpt4o-mini. embedding 모델들은 다 같은 방식으로 토큰 추출
text_splitter = TokenTextSplitter(
    encoding_name="cl100k_base", # 토큰을 세는 방식 이름
    chunk_size=1500,             # chunk당 토큰 수
    chunk_overlap=200
    # seperators = ['\n', '\n\n'] 파라미터가 없음
)
start = time.time()
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서를 쪼개면서 읽는 시간 :', runtime)

문서를 쪼개면서 읽는 시간 : 3.870635747909546


In [12]:
len(document)

180

In [17]:
# chunk 글자수
# documents[0].page_content
print([len(document.page_content) for document in documents])

[1699, 1656, 1641, 1650, 1738, 1442, 1287, 1535, 1325, 1619, 1596, 1588, 1566, 1639, 1622, 1559, 1612, 1638, 1573, 1465, 1436, 1609, 1456, 1497, 1635, 1606, 1533, 1649, 1662, 1595, 1603, 1678, 1595, 1637, 1601, 1539, 1561, 1594, 1693, 1708, 1657, 1627, 1636, 1659, 1667, 1595, 1491, 1485, 1645, 1709, 1629, 1617, 1495, 1626, 1612, 1620, 1609, 1576, 1636, 1602, 1556, 1563, 1600, 1616, 1643, 1691, 1635, 1685, 1621, 1631, 1609, 1605, 1603, 1604, 1698, 1686, 1702, 1612, 1539, 1558, 1651, 2060, 1562, 1606, 1557, 1648, 1594, 1615, 1766, 1651, 1690, 1576, 1536, 1553, 1638, 1685, 1693, 1694, 1664, 1529, 1627, 1703, 1675, 1546, 1585, 1687, 1679, 1714, 1603, 1655, 1648, 1495, 1531, 1562, 1594, 1646, 1543, 1449, 1593, 1559, 1521, 1473, 1519, 1545, 1668, 1700, 1692, 1655, 1648, 1741, 1670, 1628, 1639, 1623, 1638, 1642, 1666, 1658, 1594, 1591, 1561, 1641, 1498, 1610, 1567, 1613, 1636, 1619, 1531, 1496, 1702, 1598, 1579, 1627, 1559, 1585, 1665, 1565, 1616, 1564, 1612, 1535, 1512, 1557, 1576, 1628, 165

In [19]:
# chunk 글자수 최댓값, 최솟값
print(max([len(document.page_content) for document in documents]))
print(min([len(document.page_content) for document in documents[:-1]]))

2060
1287


## 2.2 1500 글자 쪼개서 읽어오기

In [2]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('data/소득세법(법률)(제21065호)(20260102).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 문자를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=200,
    # separators=["\n\n", "\n", " ",""] # 기본값
)
# 재귀적으로 다음 순서대로 시도:
# 1. \n\n(문단구분)
# 2. \n(줄바꿈)
# 3. " "(공백)
# 4. "" - 최후에는 글자 단위로 chunking
documents=loader.load_and_split(text_splitter=text_splitter)
runtime = time.time()-start
print('문서를 1500글자 즈음으로 쪼개면서 읽는 시간 :', runtime)
print("chunk 갯수 :", len(documents))

문서를 1500글자 즈음으로 쪼개면서 읽는 시간 : 4.1963887214660645
chunk 갯수 : 193


In [5]:
# chunk들의 글자수
print([len(document.page_content) for document in documents])

[1463, 1421, 1482, 1487, 1479, 1408, 1457, 1495, 1467, 1446, 1487, 1456, 1467, 1351, 1392, 1362, 1402, 1470, 1410, 1489, 1455, 1496, 1441, 1319, 1458, 1476, 1452, 1382, 1384, 1467, 1227, 1494, 1494, 1470, 1454, 1495, 1412, 1477, 1477, 1362, 1449, 1386, 1055, 1467, 1361, 1493, 1467, 1434, 1351, 1471, 1495, 1479, 1457, 1442, 1370, 873, 1419, 1357, 1353, 1316, 1349, 1452, 1439, 1363, 1433, 1412, 1306, 1200, 1411, 1452, 1421, 1318, 1416, 1333, 1308, 1385, 1479, 1495, 1399, 1375, 1360, 1353, 1382, 1446, 1356, 1409, 1483, 1486, 1157, 1233, 1443, 1474, 1369, 1439, 1451, 1495, 1443, 1489, 1484, 1407, 1432, 1436, 1468, 1442, 1477, 1396, 1423, 1282, 1496, 1486, 1376, 1342, 1466, 1385, 1491, 1477, 1470, 1385, 1477, 1445, 1485, 1373, 1495, 1443, 1419, 1456, 1451, 1305, 1454, 1411, 1443, 1488, 1404, 1419, 1339, 1451, 1288, 1450, 1481, 1419, 1369, 1479, 1480, 1461, 1414, 1419, 1463, 1481, 1486, 1387, 1485, 1448, 1367, 1364, 1391, 1446, 1414, 1414, 1414, 1473, 1417, 1474, 1419, 1342, 1406, 1338, 1138

In [7]:
print(max([len(document.page_content) for document in documents]))
print(min([len(document.page_content) for document in documents][:-1]))

1496
873


# 3. 쪼갠 문서를 임베딩 -> 벡터 데이터베이스 저장
- 임베딩 모델 : upstage의 solar-embedding-1-large-passage
- 벡터데이터베이스(벡터 store) : chroma

In [3]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(
    model="solar-embedding-1-large-passage"
)

In [5]:
# embed_query() 한 문자열을 임베딩 벡터로 전환한 list를 return
len(embedding.embed_query("소득세법은 다음과 같다"))

4096

In [6]:
embedding_vector = embedding.embed_documents( # 여러 문자열을 임베딩 벡터로 
    [
        "소득세법은 어쩌구",
        documents[0].page_content
    ]
)

In [7]:
print(len(embedding_vector), len(embedding_vector[0]), len(embedding_vector[1]))
print(embedding_vector[0][:10])

2 4096 4096
[0.00742340087890625, 0.0136260986328125, -0.0213165283203125, 0.019256591796875, -0.0172882080078125, 0.003154754638671875, -0.01328277587890625, -0.0156097412109375, -0.00691986083984375, -0.00238800048828125]


In [8]:
%%time
from langchain_chroma import Chroma
# 데이터 처음 저장할 때
# database = Chroma.from_documents(
#     documents=documents, # chunk
#     embedding=embedding, # 임베딩 객체
#     collection_name="tax-collection", # 생략시 이름 랜덤
#     persist_directory='./chroma_upstage'   # 생략시 로컬DB에 저장 안 되어 프로그램 종료시 DB 제거됨
# )
# 이미 저장된 vector DB(store)를 사용할 때
database = Chroma(
    embedding_function=embedding,
    collection_name='tax-collection',
    persist_directory='./chroma_upstage'
)

CPU times: total: 4.34 s
Wall time: 34 s


In [35]:
results = database._collection.get(include=['embeddings', 'documents','metadatas'])
print("데어터 수:",len(results['ids']))
print('문서 임베딩 차원 수 :', len(results['embeddings'][0]))
print("1번째 임베딩 샘플 :", results['embeddings'][1])
print("1번째 원본 :", results['documents'][1][:50])
print('1번쨰 metadatas :', results['metadatas'][1])

데어터 수: 180
문서 임베딩 차원 수 : 3072
1번째 임베딩 샘플 : [ 0.01991478 -0.01470464 -0.00057961 ...  0.0058937  -0.03365059
 -0.00657188]
1번째 원본 : . 구성원 간 이익의 분배비율이 정하여져 있지 아니하나 사실상 구성원별로 이익이 분배되는 
1번쨰 metadatas : {'source': './data/소득세법(법률)(제21065호)(20260102).docx'}


# 4. vector DB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [9]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieval_docs = database.similarity_search(query=query,
                                           k=2) # 기본 k값은 4

In [43]:
# retrieval_docs

In [12]:
# print("\n\n--\n\n".join([doc.page_content for doc in retrieval_docs]))
retrieval_doc = "\n\n--\n\n".join([doc.page_content for doc in retrieval_docs])

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변 생성 -1

In [10]:
from langchain_openai import ChatOpenAI
load_dotenv()
llm=ChatOpenAI(model="gpt-4.1-nano")

In [13]:
# prompt = f"""[identity]
# - 당신은 최고의 한국 소득세 전문가입니다
# - [context]를 참고하여 사용자의 질문에 답변해 주세요
# [context]의 내용은 다음과 같아요
# {retrieval_doc}
# 질문:{query}"""

prompt = f"""너는 대한민국 세법(특히 소득세법)에 특화된 법령 분석 AI다.
반드시 아래 원칙을 따라 질문에 답한다:
1. 제공된 문서(Context)에 근거한 내용만 답변한다.
2. 문서에 명시되지 않은 내용은 추론하거나 일반화하지 않는다.
3. 답변에는 관련 조문 번호(조·항·호)를 반드시 명시한다.
4. 문서에서 근거를 찾을 수 없는 경우, “제공된 소득세법 문서에는 해당 내용이 명시되어 있지 않습니다.”라고 답한다.
5. 실무적 조언이나 해석이 필요한 경우에도 조문을 우선 인용하고, 해석은 보조적으로만 제시한다.
[context]의 내용은 다음과 같다
{retrieval_doc}
질문:{query}"""

In [14]:
ai_message = llm.invoke(prompt)

In [15]:
ai_message.usage_metadata

{'input_tokens': 1957,
 'output_tokens': 620,
 'total_tokens': 2577,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [16]:
print(ai_message.content)

연봉 5천만원인 직장인의 소득세 산출을 위해 기본 공제, 근로소득공제, 해당 공제 항목 등을 고려하여 계산하겠습니다.

1. 총급여액: 50,000,000원

2. 근로소득공제(제47조):
- 공제액은 최대 2,000만원까지 제공
- 연봉 5천만원은 2천만원 공제액으로 제한됨
- 따라서 근로소득공제액 = 20,000,000원

3. 과세표준 계산:
- 과세표준 = 총급여액 - 근로소득공제액 = 50,000,000원 - 20,000,000원 = 30,000,000원

4. 소득세 계산:
- 대한민국의 소득세 세율은 누진세율로, 2023 기준으로 다음과 같습니다:

| 과세표준 구간 | 세율 | 누진공제액 |
|--------------|-------|------------|
| 1,200만원 이하 | 6% | 0원 |
| 1,200만원 초과 ~ 4,600만원 이하 | 15% | 1,080,000원 |
| 4,600만원 초과 ~ 8,800만원 이하 | 24% | 2,540,000원 |
| 8,800만원 초과 ~ 1.5억 이하 | 35% | 5,530,000원 |
| 1.5억 초과 | 38% | 9,220,000원 |

- 과세표준 3,000만원은 1,200만원 초과 4,600만원 이하 구간에 해당합니다.

- 세율: 15%
- 누진공제액: 1,080,000원

5. 세액 계산:
세액 = (과세표준 × 세율) - 누진공제액
= (30,000,000원 × 15%) - 1,080,000원
= 4,500,000원 - 1,080,000원
= 3,420,000원

6. 주민세 포함:
일반적으로 주민세 10%가 부과되므로
총 소득세 = 3,420,000원 + (3,420,000원 × 10%) = 3,420,000원 + 342,000원 = 3,762,000원

**최종 소득세 예상 금액은 약 3,762,000원입니다.**

참고로, 기타 인적공제(자녀세액공제 등), 보험료공제, 연금계좌공제 등이 있다면 최종 세액은 추가로 감액될 수 있습니다. 
자세한 항목을 알려주시면 더 정확

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변 생성 -2

In [59]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
load_dotenv()
llm=ChatOpenAI(model="gpt-4.1-nano")
promptTemplate=ChatPromptTemplate([
    ("system", "당신은 최고의 한국 소득세 전문가입니다."),
    ("human", f"""다음 문맥을 참고하여 질문에 답변하세요.
    답을 모르면 모른다고 말하세요.
    최대 3문장으로 간결하게 답변하세요.
    질문 : {{question}}
    문맥 : {{context}}
    답변 : """)
])
promptTemplate

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 최고의 한국 소득세 전문가입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='다음 문맥을 참고하여 질문에 답변하세요.\n    답을 모르면 모른다고 말하세요.\n    최대 3문장으로 간결하게 답변하세요.\n    질문 : {question}\n    문맥 : {context}\n    답변 : '), additional_kwargs={})])

In [60]:
prompt = promptTemplate.invoke({
    'context':retrieval_doc, # retrival_docs보다 추천
    'question' : query
})

In [62]:
llm.invoke(prompt)

AIMessage(content='연봉 5천만원인 직장인의 소득세는 일반적인 과세 표준과 공제액에 따라 계산해야 하며, 세율에 따라 대략 7~15% 정도가 될 것으로 예상됩니다. 그러나 정확한 금액은 총급여액, 공제항목, 세액공제 여부에 따라 달라지므로 구체적인 계산이 필요합니다. 구체적인 계산을 위해서는 상세한 소득 및 공제 항목 정보가 필요합니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 2156, 'total_tokens': 2261, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'id': 'chatcmpl-CvauJEe45sLZRrCcwELUEfR35B5Ek', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b9ba1-5cc9-7371-ab20-55103ca70577-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2156, 'output_tokens': 105, 'total_tokens': 2261, 'input_token_details': {'audio': 0, 'cache_read': 0

In [63]:
# 위의 예제를 langchain으로 답변생성
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(promptTemplate.invoke({
    'context':retrieval_doc,
    'question' : query
})))

'연봉 5천만원인 직장인의 소득세는 정확한 계산을 위해 근로소득공제와 기본공제, 세율 등 추가 정보가 필요합니다. 그러나 일반적으로 5천만원의 연봉에 대한 근로소득세는 약 600만원 내외가 예상됩니다. 자세한 세금액은 개인의 공제 조건에 따라 달라질 수 있습니다.'

# 6. langchain으로 답변 생성

In [64]:
# 위의 예제를 langchain으로 답변생성
rag_chain = promptTemplate | llm | output_parser
rag_chain.invoke({'context':retrieval_doc,'question':query})

'연봉 5천만원인 직장인의 소득세는 정확한 세율 계산이 필요하나, 대략적으로 10~15% 수준입니다. 이는 근로소득공제와 세율 구간에 따라 달라지며, 구체적 계산은 세액공제, 근로소득공제 및 지방소득세 등을 고려해야 합니다. 정확한 세액을 알고 싶으면 세무사와 상담하는 것을 추천드립니다.'

## langchain 전달
    smith.langchain.com 에서 key 생성 후 .env에 LANGCHAIN_API_KEY 추가

In [23]:
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

# 1. LLM과 임베딩 초기화
load_dotenv()
llm = ChatUpstage(model="solar-pro2")
embedding = UpstageEmbeddings(model="solar-embedding-1-large-passage")
# 2. vector store load
vectorstore = Chroma(
    embedding_function=embedding,
    collection_name="tax-collection",
    persist_directory="./chroma_upstage/"
)
# 3. Retriever 생성
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)
# 4. 프롬프트 템플릿
template = f"""당신은 최고의 한국 소득세 전문가입니다.
다음 문맥을 참고하여 질문에 답하세요
답을 모르면 모른다고 답하세요
최대 3문장으로 간결하게 답변하세요.
질문:{{query}}
문맥:{{context}}
답변:"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 document를 텍스트로 변환하는 함수
def format_documents(documents):
    return "\n\n--\n\n".join([doc.page_content for doc in documents])

In [24]:
# 6. RAG 체인 구성(LCEL 방식)
from langchain_core.runnables import RunnablePassthrough # {"query":"~"}=>"~"
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문 그대로 전달
    }
    | prompt # prompt에 cdontext와 query 변수 주입
    | llm 
    | StrOutputParser()
)
# 7. 실행
query ="연봉 5천만원인 직장인의 소득세는 얼마인가요?"
rag_chain.invoke(query)

'연봉 5천만원인 직장인의 소득세는 근로소득공제, 세율, 추가 공제 항목(자녀세액공제 등)에 따라 달라집니다. 제공된 문맥만으로는 정확한 세액 계산이 불가능하며, 추가 정보(가족 수, 연금계좌 납입액 등)가 필요합니다. 정확한 계산을 위해서는 국세청 홈택스 또는 전문가 상담이 필요합니다.  \n\n(참고: 근로소득공제 후 과세표준에 세율 적용, 자녀세액공제 등 추가 공제 가능)'